# 6 – SOTA Benchmarks (extended comparison)

Extended benchmark families for Bioinformatics submission:
1. Strong tabular baselines
2. Fixed-prior graph controls
3. PAM50 centroid classifier
4. LightGBM + CatBoost
5. Simple GCN baseline
6. 291-gene signature tabular
7. Random/rewired prior control
8. GraphSAGE baseline

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from intergate.config import CFG
from intergate.utils import set_seed, set_all_seeds, cleanup_memory
from intergate.data import (
    load_expression_and_metadata, prepare_genes, encode_labels,
    cohort_split, scale_features, apply_connected_only,
    make_dataloaders, build_regulator_features, ExpressionDataset,
)
from intergate.graph import build_backbone
from intergate.graph_cache import get_or_build_backbone, get_or_build_Xh
from intergate.losses import compute_metrics_full, compute_class_weights_balanced
from intergate.training import predict_proba_xh_mode
from intergate.pruning import export_pruned_graph
from intergate.ablation import (
    AblationConfig, load_bundle, build_pruned_model_from_bundle,
    register_runtime, get_full_gene_matrix_and_genes,
    run_single_seed, run_ablation,
)

from intergate.ablation import (
    AblationConfig, build_model_from_cfg, make_prior_for_cfg,
    run_single_seed, run_ablation, register_runtime,
)

set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


In [ ]:
from intergate.benchmarks import (
    run_tabular_baselines_bioinfo,
    run_fixed_prior_graph_controls_bioinfo,
    run_centroid_baselines_bioinfo,
    run_sota_tabular_bioinfo,
    run_simple_gcn_bioinfo,
    build_sota_summary_table,
    run_signature291_tabular_benchmarks_bioinfo,
    run_graphsage_bioinfo,
    build_requested_benchmarks_summary,
)


## 1) Data pipeline

In [ ]:
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV, CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL, label_col=CFG.LABEL_COL, cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)


In [ ]:
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg, cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=False, use_omnipath=CFG.USE_OMNIPATH, use_huri=CFG.USE_HURI,
)
print(f"Backbone: {edge_index.shape[1]} aristas, {len(genes_kegg)} genes")


In [ ]:
train_idx, val_idx, test_idx = cohort_split(
    cohort, y, train_cohort_frac=0.80, val_size=CFG.VAL_SIZE, seed=CFG.SEED,
)
Xs_gene = scale_features(X_df_kegg, train_idx, mode=CFG.SCALE_MODE, use_quantile=CFG.USE_QUANTILE)

if CFG.CONNECTED_ONLY:
    Xs_gene, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only(
        Xs_gene, edge_index, edge_weight, edge_type, genes_kegg,
    )


In [ ]:
Xs_graph, graph_feat_names = get_or_build_Xh(
    Xs_gene, genes_kegg, edge_index,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    stats=CFG.REG_STATS, min_targets=CFG.REG_MIN_GENES,
    max_regulators=CFG.REG_MAX_REGULATORS,
)


In [ ]:
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene, Xs_graph, y, train_idx, val_idx, test_idx, batch_size=CFG.BATCH_SIZE,
)


In [ ]:
edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

register_runtime(
    Xs_gene=Xs_gene, genes_kegg=genes_kegg, X_h=Xs_graph,
    y=y, n_classes=n_classes, label_map=label_map,
    train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
    edge_index_t=edge_index_t, edge_weight_t=edge_weight_t, edge_type_t=edge_type_t,
)


In [ ]:
from intergate.benchmarks import (
    run_tabular_baselines_bioinfo,
    run_fixed_prior_graph_controls_bioinfo,
    run_centroid_baselines_bioinfo,
    run_sota_tabular_bioinfo,
    run_simple_gcn_bioinfo,
    run_graphsage_bioinfo,
    run_signature291_tabular_benchmarks_bioinfo,
    build_sota_summary_table,
    build_requested_benchmarks_summary,
)
from intergate.benchmarks import register_runtime_benchmarks
register_runtime_benchmarks(
    Xs_gene=Xs_gene, train_idx=train_idx, val_idx=val_idx,
    test_idx=test_idx, y=y, label_map=label_map,
    genes_kegg=genes_kegg, n_classes=n_classes,
    edge_index=edge_index, X_h=Xs_graph, DEVICE=DEVICE,
    BATCH_SIZE=CFG.BATCH_SIZE,
    make_prior_for_cfg=make_prior_for_cfg,
    AblationConfig=AblationConfig,
)

## 2) Strong tabular baselines

In [ ]:
df_tab = run_tabular_baselines_bioinfo(random_state=1234, out_csv="./bioinfo_tabular_baselines.csv")
display(df_tab.sort_values("test_macro_f1", ascending=False))


## 3) Fixed-prior graph controls

In [ ]:
df_fix = run_fixed_prior_graph_controls_bioinfo(out_csv="./bioinfo_fixed_prior_graph_controls.csv")
display(df_fix.sort_values("test_macro_f1", ascending=False))


## 4a) PAM50 centroid classifier

In [ ]:
df_cent = run_centroid_baselines_bioinfo(out_csv="./bioinfo_centroid_benchmarks.csv")
display(df_cent)


## 4b) LightGBM + CatBoost

In [ ]:
df_sota_tab = run_sota_tabular_bioinfo(random_state=1234, out_csv="./bioinfo_sota_tabular.csv")
display(df_sota_tab.sort_values("test_macro_f1", ascending=False))


## 4c) Simple GCN baseline

In [ ]:
df_gcn = run_simple_gcn_bioinfo(out_csv="./bioinfo_simple_gcn.csv")
display(df_gcn)


## 4d) Consolidated SOTA summary

In [ ]:
df_sota = build_sota_summary_table(out_csv="./bioinfo_sota_summary.csv")
display(df_sota)


## 5a) 291-gene signature tabular benchmarks

In [ ]:
df_291 = run_signature291_tabular_benchmarks_bioinfo(out_csv="./bioinfo_signature291_tabular.csv")
display(df_291.sort_values("test_macro_f1", ascending=False))


## 5c) GraphSAGE baseline

In [ ]:
df_sage = run_graphsage_bioinfo(out_csv="./bioinfo_graphsage.csv")
display(df_sage)


## 5d) Requested benchmarks summary

In [ ]:
df_req = build_requested_benchmarks_summary(out_csv="./bioinfo_requested_extra_benchmarks_summary.csv")
display(df_req)


# Fin